# [2026年4月22日] LLMって何？実演編

本ノートブックでは, HuggingFaceというサイトにあるGeminiモデルを使用して, LLMの動作を体感しよう.

## LLMのイメージ

大規模言語モデル(large language model; LLM)とは, 膨大なテキストデータから得た「知識」に基づいて, **所与の入力に対して, 尤もらしい(よくある)回答を返す**ように訓練されたモデル(仕組み)です.

ChatGPTやGemini, Claudeなど「生成AI」の構築と進化で活用されている技術です.

### 言語モデル

- 「次に続く単語」の確率をモデル化しているのが現在の言語モデルです.
    - 例えば「きょうの天気は」と言われて「ピザです」と答えることは常識的あり得ないが「晴れです」と答えることは妥当です.
    - 言い換えれば「きょうの天気は」という文の続きに「ピザ」という単語が来る確率は低いが「晴れ」という単語が来る確率は高いです.
    - こういう「よくある」続きが出てくるように, モデルを訓練する必要があります.

### 大規模であるとは

- 事前に構築した単語全体の集合(語彙)の中で, 所与の文章に続く単語としてどれが「よくあるか」を学習するには, あるあるパターンをモデルに「習得」させる必要があります. これが「機械学習」の「学習」に相当するものです.
- 大規模であるとは, その「習得」いわば「学習」のために使われているデータの量が多いということです. 各所に溢れている膨大な量の文章資源(コーパス)から, 文章のよくあるパターンを「学習」させます. 一般的には**数百万冊分の書籍相当の文章量**から学習しています.

## 作成者
坪井 一馬
- 横浜国立大学 理工学部 化学・生命系学科 化学EP 4年生
- 化学と情報科学を融合したケモインフォマティクスの研究をしている. 特に, 有機化合物データベースや特許情報を扱う観点でLLMを日々扱う.
- 東京大学松尾・岩澤研究室のLLM講座2025基礎編/応用編を修了済み.
    - 基礎編の修了率47%, 応用編の修了率27%.

## 注意
- 難しいので, 学術的な厳密性や数理的背景には踏み込みません.
- 皆さんの手で動作するには色々準備が必要なので, **ここでは坪井による実演と資料共有にとどめます**.
    - 私の実行では, 有料で高性能なGPUに課金して高速に実施していますが, 無料版であればおそらく30分くらい待機が必要です. 落ちてしまうこともあります.
    - **Lumosに入ってくれたら, 皆さんの手で, 皆さんのPCで色々いじる機会を積極的に設けますのでお楽しみに!**
- 本資料にはGoogle Geminiを使用して構築している部分がありますが, すべて坪井の目を通しており, 誤りがないことを確認済みです.

## 0. APIキーの設定
- 今回はWeb上で公開されているモデルを動かしてみます.
- Geminiモデルを使うには, 事前に取得したAPIキーと呼ばれるものを入力する必要があります. 簡単にいうと**利用者として認定されていることの証明**です.
- 事前に取得の必要がありますが, 今は実演なので坪井のほうで登録済みのものを読ませます.

In [1]:
!pip install -q transformers accelerate

import torch
from transformers import pipeline
from google.colab import userdata

# Hugging Face Tokenの設定
try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = None
    print("HF_TOKENが設定されていません。")

def load_hf_model(model_id):
    print(f"{model_id} をロード中...")
    return pipeline(
        "text-generation",
        model=model_id,
        model_kwargs={"torch_dtype": torch.bfloat16},
        device_map="auto",
        token=hf_token
    )

model_standard_id = "google/gemma-2-2b-it"

print("--- gemma-2-2b-itのモデルをロード ---")
pipe_standard = load_hf_model(model_standard_id)

--- gemma-2-2b-itのモデルをロード ---
google/gemma-2-2b-it をロード中...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

## 1. どういう感じで動く？
- モデルに対して入力する文章を「プロンプト」といいます. モデルは「プロンプト」に対して**よくある回答**を返してくれます.
  - 出力は毎回変わります.
  - 一般論として「よくある」ことが「正しい」とは限りません. 例えば, 以下では「横浜国立大学」を説明させますが, おそらく学部名を間違えて, 文学部や医学部と出てきます. これはなぜかというと, 一般的な「国立大学」ではそういう学部があるからです.

In [2]:
# 実行するプロンプト
prompt = "ホワイトクリームとミックスチョコ, レーズンを2枚のクッキーで挟むお菓子を大学祭で販売しますが、盛り付け方のコツはありますか。親切に教えて"

# 9b (Flash相当) モデルで生成
messages = [{"role": "user", "content": prompt}]
outputs = pipe_standard(
    messages,
    max_new_tokens=512,
    do_sample=True,
    temperature=1.0,
)
response_text = outputs[0]["generated_text"][-1]["content"]

print(f"--- プロンプト ---\n{prompt}\n")
print(f"--- LLMの回答 ---\n{response_text}")

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- プロンプト ---
ホワイトクリームとミックスチョコ, レーズンを2枚のクッキーで挟むお菓子を大学祭で販売しますが、盛り付け方のコツはありますか。親切に教えて

--- LLMの回答 ---
大学祭で販売する、ホワイトクリームとミックスチョコ、レゼンを挟むクッキーの盛り付け、素敵なアイデアですね！ 

**盛り付けのポイント**

1. **華やかさとバランス:**  
    * **ホワイトクリームをポイントに:** 大きな輪郭を描いて、華やかさを出す。
    * **ミックスチョコでアクセント:**  ホワイトクリームの周りで、ミックスチョコを細かく砕いて配置する。
    * ** レーズンをトッピング:** レーズンをトッピングして華やかさをプラス。

2. **サイズと層感:**
   * **小さめサイズのクッキー:**  様々なサイズが並んで、華やかさと食べやすさを両立させる。

3. **形状と配置:**
   * **ハート形:**  チョコレートをハート型に配置して可愛く仕上げる。 
   * **並列:**  クッキーを並設で並べ、シンプルながらもスタイリッシュな雰囲気を出す


**具体的な盛り付け例**

* **クラシックスタイル:**  ホワイトクリーム（丸めて、少し盛り高めに）とミックスチョコ（少し多めに）を交互に挟む。レザンのフレークを少しだけ乗せる。

* **アートスタイル:**  ホワイトクリームを丸い形を描き、チョコレートを沿って流して、レザンのフレークを添える。

* **ボリュームスタイル:**  ホワイトクリームとミックスチョコを交互に盛りつけ、全体を少し盛り上げて厚く仕上げる。


 **ポイント**

* **形は自由:**  様々な形でお客さんの目を惹きつける！ 
* **食べやすさ:**  食べやすさのため、クッキーのサイズや形、盛り付け方を工夫する。
* **バランス:**  見た目だけでなく、バランスも大切。

**その他**

* **トッピング:**  ナッツやドライフルーツなどをトッピングして、さらに華やかさをプラス。 
* **包装:**  クッキーを可愛く包み、持ちやすくするため、パッケージを工夫する。


大学祭に販売するクッキーは、見た目だけでなく、味も大切です！ぜひ、これらのポイ

## 2. パラメータによる変化
- こんなのGeminiの画面でやるのと変わらないと思ったアナタへ
    - LLMエンジニアリングの世界では, **モデルから所望の回答を引き出す**ために. **設定値を調整する**とか, **他のモデルと組み合わせる**ことがあります. 入力を変えるだけではありません.
    - LLMによる生成結果の評価方法として, 実際の研究では, 人手で良いものと悪いものを評価する方法や, 別のLLMによって評価する方法(LLM-as-a-judge)が行われます.
    - コードベースでの**プログラミングによる工夫もできます**ので, それに関してはLumosに入ってくれたら体験できる機会を提供する予定です.
- LLMの振る舞いを決める指標として, 様々な**数値的パラメータ**があります.
    - どのくらいの長さまで生成をさせるか, 途中まで候補を留め置いておくか否かなど.
    - ここでは, 回答の「いい加減さ」を制御する温度パラメータ`temperature`を調整してみます. 温度パラメータは0.0から2.0の範囲で指定可能です.
        - **低い値 (0.1程度)**: 常に最も確率の高い言葉を選び, 論理的で安定した回答になります. ビジネス的には最適だがつまらない？
        - **中程度の値 (1.0程度)**: 確率的に尤もらしいものを出します. 事実ベースで間違ったり, おかしなものを生成することもあります. ここまでの実演では1.0にしていました.
        - **高い値 (2.0程度)**: 生成が崩壊することもあります. 面白おかしい文章を生成したいならば有効ではありますが...
- 私の事前テストで不適切な内容が出てしまったため「不適切な内容は出力しないこと」というプロンプトを明示的に入れます. これだけでもかなり効きます.

In [3]:
def test_temp_hf(pipe, temp_value):
    creative_prompt = "ホワイトクリームとミックスチョコ, レーズンを2枚のクッキーで挟むお菓子を大学祭で販売しますが、盛り付け方のコツはありますか。親切に教えて"
    messages = [{"role": "user", "content": creative_prompt}]

    outputs = pipe(
        messages,
        max_new_tokens=512,
        do_sample=True if temp_value > 0 else False,
        temperature=temp_value if temp_value > 0 else None
    )

    print(f"=== Temperature: {temp_value} ===")
    print(outputs[0]["generated_text"][-1]["content"])

In [4]:
print("堅実な回答")
test_temp_hf(pipe_standard, 0.1)

print("-" * 30)

print("創造的な回答")
test_temp_hf(pipe_standard, 1.0)

print("-" * 30)

print("生成が崩壊するかも")
test_temp_hf(pipe_standard, 2.0)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


堅実な回答


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Temperature: 0.1 ===
大学祭で販売するホワイトクリームとミックスチョコ、レズンのクッキー、盛り付け方のコツをいくつかご紹介します！

**1.  バランスと美しさ**

* **クッキーのサイズ:**  均等にカットして、食べやすいサイズにしましょう。
* **盛り付けの高さ:**  クッキーの厚みとクリームの量を考慮して、バランス良く盛り付けましょう。
* **隙間:**  クッキー同士の隙間を少し残すことで、華やかさを演出できます。

**2.  視覚的な魅力**

* **色使い:**  ホワイトクリームとミックスチョコ、レズンの色を組み合わせ、視覚的に魅力的な盛り付けを目指しましょう。
* **デコレーション:**  クッキーの表面に、チョコレートペンで星やハートなどを描いたり、レズンの粉をまぶしたりして、デコレーションを加えてみましょう。
* **トッピング:**  ナッツやドライフルーツなどをトッピングすることで、食感と彩りをプラスできます。

**3.  販売促進**

* **価格設定:**  材料費や時間などを考慮して、価格を適切に設定しましょう。
* **ネーミング:**  ユニークで覚えやすい名前をつけ、販売促進に役立てましょう。
* **説明:**  商品の魅力を伝える説明書きを添えて、販売促進を図りましょう。

**具体的な盛り付け例**

* **シンプル:**  クッキーを2枚重ね、ホワイトクリームを挟み、ミックスチョコをトッピング。
* **華やか:**  クッキーを2枚重ね、ホワイトクリームを挟み、レズンの粉をまぶし、チョコレートペンで星やハートを描き、ナッツをトッピング。
* **ボリューム:**  クッキーを3枚重ね、ホワイトクリームを挟み、ミックスチョコをトッピング、レズンの粉をまぶし、ナッツをトッピング。

**その他**

* **衛生管理:**  クッキーをきれいに保ち、衛生的に販売しましょう。
* **持ち運び:**  クッキーを安定して持ち運べるように、箱や袋を用意しましょう。
* **試食:**  試食をさせて、商品の魅力を伝えましょう。

**ポイント**

* **お客様の好み:**  大学祭のターゲット層を考慮し、盛り付けを調整しましょう。
* **季節感:**  季節の食材をうま

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Temperature: 1.0 ===
大学祭の販売、頑張ってくださいね！ ホワイトクリームとミックスチョコ、レジンを挟むクッキーの盛り付け、工夫次第で魅力的に仕上がりますよ！ 

**盛り付けのポイント**

1. **断面重視:**  クッキーの断面が美しいものをおすすめします。
    * **均一な厚さ:**  薄い層を複数枚重ねて、厚くまとめてあると、食べやすくなります。
    * **美しく整えた形:**  クッキーの切り方、挟む具材の配置を工夫することで、華やかさを演出できます。
    * **カットの角度:**  端を斜めにすることで、よりおしゃれな印象になります。
2. **色彩とバランス:** 
    * **ホワイトクリーム:**  クリームが全体に広がってるイメージ。
       * **ポイント:**  クリームを、軽く押えるように配置することで、層分かれて見えるイメージを演出します。
    * **ミックスチョコ:**  チョコが表面に隠れることを避けて、層に表示します。
3. **レザンの魅力を発揮:**
    *  **アクセント:**  レザンの色は、ホワイトクリームと組み合わせると、より引き締める効果が期待できます。
    * **大きめなサイズ:**  レザンを乗せることで、クッキーの視認性を向上させます。

**盛り付け例**

* **シンプルバージョン:** 
    *  クッキーを2枚重ね、ホワイトクリームを中央に挟みます。
    *  ミックスチョコとレザンの層を、真ん中のホワイトクリームの上に並べます。
    *  断面が美しく、シンプルで食べやすいです。
* **可愛らしさを演出:** 
    *  クッキーを4枚に分割し、ホワイトクリームを2枚ずつ重ね、真ん中に挟みます。
    *  ミックスチョコとレザンの層を、ホワイトクリームの上と下の2箇所に取り付けます.
    *  カットして、かわいらしい形に。

**その他補足**

* **温度管理:**  クッキーの温度は、クリームが固まらないように注意してください。
* **包装:**  オーナメントやリボンなどの装飾、ミニチュアでクッキーを包装することも可能です。
* **試食:**  販売前には、友人に試食してもらい、意見

## 4. 保存
- 何か実行をしたら, それを再現できるように保存をしておくことは重要です.
- 皆さんに共有をするため, GitHub Gistというサイトへのアップロード, HTMLとしての出力を用意します. これは事務的なコードです.

In [5]:
import json
from google.colab import _message
from google.colab import userdata
import requests

# --- GitHub Gist & HTML出力設定 ---
save_name = "260423_What_is_LLM_test"
save_name_gist = "260423_What_is_LLM_test.ipynb"
# ----------

# 1. 現在のノートブックのJSONデータを取得
notebook_json_raw = _message.blocking_request('get_ipynb', request='', timeout_sec=5)
ipynb_dict_raw = notebook_json_raw['ipynb']

# --- GitHub Gistへの保存 ---
print("---> GitHub Gistへの保存を開始します --->")

try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    github_token = None
    print(f"GitHub TOKENの取得に失敗しました: {e}")
    print("ColabのSecretsに 'GITHUB_TOKEN' を設定してください。")

if github_token is None:
    print("GitHub TOKENが設定されていないため、Gistの作成をスキップします。")
else:
    filename_gist = save_name_gist
    description_gist = "LLMって何かを解説する"
    is_public_gist_str = 'yes'
    is_public_gist = True if is_public_gist_str == 'yes' else False

    url_gist = 'https://api.github.com/gists'
    headers_gist = {
        'Authorization': f'token {github_token}',
        'Accept': 'application/vnd.github.v3+json'
    }

    ipynb_dict_for_gist = json.loads(json.dumps(ipynb_dict_raw)) # Deep copy

    if 'metadata' not in ipynb_dict_for_gist:
        ipynb_dict_for_gist['metadata'] = {}
    if 'widgets' not in ipynb_dict_for_gist['metadata']:
        ipynb_dict_for_gist['metadata']['widgets'] = {}

    if "application/vnd.jupyter.widget-state+json" not in ipynb_dict_for_gist['metadata']['widgets'] or not isinstance(ipynb_dict_for_gist['metadata']['widgets']["application/vnd.jupyter.widget-state+json"], dict):
        ipynb_dict_for_gist['metadata']['widgets']["application/vnd.jupyter.widget-state+json"] = {}

    ipynb_dict_for_gist['metadata']['widgets']["application/vnd.jupyter.widget-state+json"]["state"] = {}

    notebook_content_str_gist = json.dumps(ipynb_dict_for_gist, ensure_ascii=False, indent=4)

    data_gist = {
        'description': description_gist,
        'public': is_public_gist,
        'files': {
            filename_gist: {
                'content': notebook_content_str_gist
            }
        }
    }

    print("\nGistを作成中...")
    try:
        response_gist = requests.post(url_gist, headers=headers_gist, data=json.dumps(data_gist))
        response_gist.raise_for_status()

        gist_data = response_gist.json()
        print(f"Gistが正常に作成されました！\nURL: {gist_data['html_url']}")
    except requests.exceptions.HTTPError as err:
        print(f"HTTPエラーが発生しました: {err}")
        print(f"レスポンス: {response_gist.text}")
    except Exception as err:
        print(f"Gistの作成中にエラーが発生しました: {err}")

---> GitHub Gistへの保存を開始します --->

Gistを作成中...
Gistが正常に作成されました！
URL: https://gist.github.com/Tsuboi-coder/0c44096b39327cef9edf7ef9d0c3f432


In [6]:
print("\n--- HTML出力を開始します ---> ")

ipynb_dict_for_html = json.loads(json.dumps(ipynb_dict_raw)) # ディープコピーを作成

# KeyError: 'state' 回避のため、メタデータからwidgets情報をクリア
if 'metadata' in ipynb_dict_for_html:
    if 'widgets' in ipynb_dict_for_html['metadata']:
        del ipynb_dict_for_html['metadata']['widgets'] # widgetsキー自体を削除

# 指定した名前でipynbファイルとして保存
ipynb_filename_html = f'{save_name}.ipynb'
with open(ipynb_filename_html, 'w', encoding='utf-8') as f:
    json.dump(ipynb_dict_for_html, f, ensure_ascii=False, indent=4)

# 保存したipynbファイルをHTMLに変換
!jupyter nbconvert --to html {ipynb_filename_html}

print(f'\n--- 完了 ---\nHTML出力ファイル: {save_name}.html が作成されました。左側のファイルメニューからダウンロードしてください。')


--- HTML出力を開始します ---> 
[NbConvertApp] Converting notebook 260423_What_is_LLM_test.ipynb to html
[NbConvertApp] Writing 319092 bytes to 260423_What_is_LLM_test.html

--- 完了 ---
HTML出力ファイル: 260423_What_is_LLM_test.html が作成されました。左側のファイルメニューからダウンロードしてください。
